# Preliminary Data Analysis of Cold Storage Room Environmental Data

## Import of Required Packages

In [1]:
import os
import pandas as pd
import numpy as np
from scipy.signal import find_peaks
import stumpy

## Environmental Variables

In [2]:
PATH_PREPROCESSED = '../Data/Preprocessed'
FILE_OVERVIEW = '../Data/overview.csv'

# detection of preprocessed datasets
datasets_preprocessed = [file[0:8] for file in os.listdir(PATH_PREPROCESSED) if os.path.isfile(os.path.join(PATH_PREPROCESSED, file))]

## Statistical Analysis

In [4]:
statistics = pd.DataFrame(data=None, columns=['Sensor', 'Temperature min', 'Temperature max', 'Temperature mean', 'Temperature std', 'Temperature q25', 'Temperature median', 'Temperature q75', 'Humidity min', 'Humidity max', 'Humidity mean', 'Humidity std', 'Humidity q25', 'Humidity median', 'Humidity q75'])

for dataset in datasets_preprocessed:
    # construct the full path to the preprocessed CSV file for the current dataset
    path_file = os.path.join(PATH_PREPROCESSED, f"{dataset}_preprocessed.csv")
    
    # read the CSV file into data frame, using ';' as the separator and the first row as header
    data = pd.read_csv(path_file, sep=";", header=0)
    
    min_values = data[['Temperature', 'Humidity']].min()
    max_values = data[['Temperature', 'Humidity']].max()
    mean = data[['Temperature', 'Humidity']].mean()
    std = data[['Temperature', 'Humidity']].std()
    quantiles = data[['Temperature', 'Humidity']].quantile([0.25, 0.75])
    median = data[['Temperature', 'Humidity']].median()
    
    statistics.loc[len(statistics.index)] = [dataset, min_values['Temperature'], max_values['Temperature'], mean['Temperature'], std['Temperature'], quantiles['Temperature'].loc[0.25], median['Temperature'], quantiles['Temperature'].loc[0.75], min_values['Humidity'], max_values['Humidity'], mean['Humidity'], std['Humidity'], quantiles['Humidity'].loc[0.25], median['Humidity'], quantiles['Humidity'].loc[0.75]]
 
statistics = statistics.transpose()
   
if 'datasets_overview' not in globals():
    datasets_overview = pd.read_csv(FILE_OVERVIEW, sep=";", header=0, index_col=0)
if 'Temperature min' in datasets_overview.index:
    datasets_overview = datasets_overview.drop(index=['Temperature min', 'Temperature max', 'Temperature mean', 'Temperature std', 'Temperature q25', 'Temperature median', 'Temperature q75', 'Humidity min', 'Humidity max', 'Humidity mean', 'Humidity std', 'Humidity q25', 'Humidity median', 'Humidity q75'])

datasets_overview = datasets_overview.merge(statistics, on='Sensor', how='left')
datasets_overview = datasets_overview.reindex(['Sensor', 'Start', 'End', 'Missing Start', 'Missing End', 'Length', 'Temperature min', 'Temperature max', 'Temperature mean', 'Temperature std', 'Temperature q25', 'Temperature median', 'Temperature q75', 'Humidity min', 'Humidity max', 'Humidity mean', 'Humidity std', 'Humidity q25', 'Humidity median', 'Humidity q75', 'Prominence', 'Peaks_Temperature', 'Peaks_Humidity', 'Peaks_Temperature Start', 'Peaks_Temperature End', 'Peaks_Humidity Start', 'Peaks_Humidity End', 'Peak Prominences Temperature', 'Peak Prominences Humidity', 'MP_Temperature min', 'MP_Temperature max', 'MP_Temperature mean', 'MP_Temperature std', 'MP_Temperature q25', 'MP_Temperature median', 'MP_Temperature q75', 'MP_Humidity min', 'MP_Humidity max', 'MP_Humidity mean', 'MP_Humidity std', 'MP_Humidity q25', 'MP_Humidity median', 'MP_Humidity q75'])
datasets_overview.to_csv(FILE_OVERVIEW, sep=";")

## Determining Local Extremes

In [13]:
local_extremes = pd.DataFrame(data=None, columns=['Sensor', 'Prominence', 'No. Peaks_Temperature', 'No. Peaks_Humidity', 'Peaks_Temperature Height min', 'Peaks_Temperature Height max', 'Peaks_Temperature Height mean', 'Peaks_Temperature Height std', 'Peaks_Temperature Height q25', 'Peaks_Temperature Height median', 'Peaks_Temperature Height q75', 'Peaks_Humidity Height min', 'Peaks_Humidity Height max', 'Peaks_Humidity Height mean', 'Peaks_Humidity Height std', 'Peaks_Humidity Height q25', 'Peaks_Humidity Height median', 'Peaks_Humidity Height q75', 'Peaks_Temperature', 'Peaks_Humidity', 'Peaks_Temperature Start', 'Peaks_Temperature End', 'Peaks_Humidity Start', 'Peaks_Humidity End', 'Peak Prominences Temperature', 'Peak Prominences Humidity'])

for dataset in datasets_preprocessed:      
    path_file = os.path.join(PATH_PREPROCESSED, f"{dataset}_preprocessed.csv")
        
    data = pd.read_csv(path_file, sep=";", header=0)
    
    for data_type in ["Temperature", "Humidity"]:
        if data_type == "Humidity":
            if dataset in ["SENSOR01", "SENSOR04", "SENSOR06"]:
                peak_prominence = 18
            elif dataset in ["SENSOR07"]:
                peak_prominence = 20.5
            elif dataset in ["SENSOR08"]:
                peak_prominence = 23
            elif dataset in ["SENSOR09"]:
                peak_prominence = 25
            else:
                peak_prominence = 20
        else:
            # if dataset in sensors_movable:
            #     peak_prominence = 2
            if dataset in ["SENSOR03"]:
                peak_prominence = 1.5
            elif dataset in ["SENSOR04", "SENSOR05"]:
                peak_prominence = 0.5
            elif dataset in ["SENSOR06"]:
                peak_prominence = 0.8
            else:
                peak_prominence = 1
        
        peaks_index, peaks_properties = find_peaks(data[data_type],  prominence=peak_prominence, distance=60)  # distance=60 since door was min. 5 min closed (helps to filter amount of peaks)
        
        globals()[f"num_peaks_{data_type.lower()}"] = len(peaks_index)
        
        peaks_df = pd.DataFrame({"peaks_index": peaks_index, "left_bases": peaks_properties["left_bases"], "right_bases": peaks_properties["right_bases"], "prominences": peaks_properties["prominences"]})
        
        filtered_peaks_df = peaks_df.loc[peaks_df.groupby(["left_bases", "right_bases"])["prominences"].idxmax()]
        
        peaks_heights = pd.Series(data[data_type].iloc[peaks_index].to_numpy() - data[data_type].iloc[peaks_properties["left_bases"]].to_numpy())
        globals()[f"peaks_{data_type.lower()}_height_min"] = peaks_heights.min()
        globals()[f"peaks_{data_type.lower()}_height_max"] = peaks_heights.max()
        globals()[f"peaks_{data_type.lower()}_height_mean"] = peaks_heights.mean()
        globals()[f"peaks_{data_type.lower()}_height_std"] = peaks_heights.std()
        globals()[f"peaks_{data_type.lower()}_height_q25"] = peaks_heights.quantile(0.25)
        globals()[f"peaks_{data_type.lower()}_height_median"] = peaks_heights.median()
        globals()[f"peaks_{data_type.lower()}_height_q75"] = peaks_heights.quantile(0.75)
        
        peaks_dates = pd.to_datetime(data.iloc[filtered_peaks_df["peaks_index"], 0] + data.iloc[filtered_peaks_df["peaks_index"], 1], format='%d.%m.%Y%H:%M:%S').dt.strftime('%Y-%m-%d %H:%M:%S').to_list()
        peaks_str = ', '.join(peaks_dates)
        
        peaks_begin = pd.to_datetime(data.iloc[filtered_peaks_df["left_bases"], 0] + data.iloc[filtered_peaks_df["left_bases"], 1], format='%d.%m.%Y%H:%M:%S').dt.strftime('%Y-%m-%d %H:%M:%S').to_list()
        peaks_str_begin = ', '.join(peaks_begin)
        
        peaks_end = pd.to_datetime(data.iloc[filtered_peaks_df["right_bases"], 0] + data.iloc[filtered_peaks_df["right_bases"], 1], format='%d.%m.%Y%H:%M:%S').dt.strftime('%Y-%m-%d %H:%M:%S').to_list()
        peaks_str_end = ', '.join(peaks_end)
        
        prominences_str = ', '.join(filtered_peaks_df["prominences"].astype(str).to_list())
        globals()[f"peaks_{data_type.lower()}"] = peaks_str
        globals()[f"peaks_{data_type.lower()}_begin"] = peaks_str_begin
        globals()[f"peaks_{data_type.lower()}_end"] = peaks_str_end
        globals()[f"prominences_{data_type.lower()}"] = prominences_str
        
    local_extremes.loc[len(local_extremes.index)] = [dataset, peak_prominence, num_peaks_temperature, num_peaks_humidity, peaks_temperature_height_min, peaks_temperature_height_max, peaks_temperature_height_mean, peaks_temperature_height_std, peaks_temperature_height_q25, peaks_temperature_height_median, peaks_temperature_height_q75, peaks_humidity_height_min, peaks_humidity_height_max, peaks_humidity_height_mean, peaks_humidity_height_std, peaks_humidity_height_q25, peaks_humidity_height_median, peaks_humidity_height_q75, peaks_temperature, peaks_humidity, peaks_temperature_begin, peaks_temperature_end, peaks_humidity_begin, peaks_humidity_end, prominences_temperature, prominences_humidity]
    
if 'datasets_overview' not in globals():
    datasets_overview = pd.read_csv(FILE_OVERVIEW, sep=";", header=0)
if 'Prominence' in datasets_overview.index:
    datasets_overview = datasets_overview.drop(index=['Prominence', 'No. Peaks_Temperature', 'No. Peaks_Humidity', 'Peaks_Temperature Height min', 'Peaks_Temperature Height max', 'Peaks_Temperature Height mean', 'Peaks_Temperature Height std', 'Peaks_Temperature Height q25', 'Peaks_Temperature Height median', 'Peaks_Temperature Height q75', 'Peaks_Humidity Height min', 'Peaks_Humidity Height max', 'Peaks_Humidity Height mean', 'Peaks_Humidity Height std', 'Peaks_Humidity Height q25', 'Peaks_Humidity Height median', 'Peaks_Humidity Height q75', 'Peaks_Temperature', 'Peaks_Humidity', 'Peaks_Temperature Start', 'Peaks_Temperature End', 'Peaks_Humidity Start', 'Peaks_Humidity End', 'Peak Prominences Temperature', 'Peak Prominences Humidity'])

datasets_overview = datasets_overview.merge(local_extremes, on='Sensor', how='left')
datasets_overview = datasets_overview.reindex(['Sensor', 'Start', 'End', 'Missing Start', 'Missing End', 'Length', 'Temperature min', 'Temperature max', 'Temperature mean', 'Temperature std', 'Temperature q25', 'Temperature median', 'Temperature q75', 'Humidity min', 'Humidity max', 'Humidity mean', 'Humidity std', 'Humidity q25', 'Humidity median', 'Humidity q75', 'Prominence', 'No. Peaks_Temperature', 'No. Peaks_Humidity', 'Peaks_Temperature Height min', 'Peaks_Temperature Height max', 'Peaks_Temperature Height mean', 'Peaks_Temperature Height std', 'Peaks_Temperature Height q25', 'Peaks_Temperature Height median', 'Peaks_Temperature Height q75', 'Peaks_Humidity Height min', 'Peaks_Humidity Height max', 'Peaks_Humidity Height mean', 'Peaks_Humidity Height std', 'Peaks_Humidity Height q25', 'Peaks_Humidity Height median', 'Peaks_Humidity Height q75', 'Peaks_Temperature', 'Peaks_Humidity', 'Peaks_Temperature Start', 'Peaks_Temperature End', 'Peaks_Humidity Start', 'Peaks_Humidity End', 'Peak Prominences Temperature', 'Peak Prominences Humidity', 'MP_Temperature min', 'MP_Temperature max', 'MP_Temperature mean', 'MP_Temperature std', 'MP_Temperature q25', 'MP_Temperature median', 'MP_Temperature q75', 'MP_Humidity min', 'MP_Humidity max', 'MP_Humidity mean', 'MP_Humidity std', 'MP_Humidity q25', 'MP_Humidity median', 'MP_Humidity q75'])
datasets_overview.to_csv(FILE_OVERVIEW, index=False, sep=";")

## Analyze Matrix Profiles

In [31]:
matrix_profiles = pd.DataFrame(data=None, columns=['Sensor', 'MP_Temperature min', 'MP_Temperature max', 'MP_Temperature mean', 'MP_Temperature std', 'MP_Temperature q25', 'MP_Temperature median', 'MP_Temperature q75', 'MP_Humidity min', 'MP_Humidity max', 'MP_Humidity mean', 'MP_Humidity std', 'MP_Humidity q25', 'MP_Humidity median', 'MP_Humidity q75'])

for dataset in datasets_preprocessed:
    # construct the full path to the preprocessed CSV file for the current dataset
    path_file = os.path.join(PATH_PREPROCESSED, f"{dataset}_preprocessed.csv")
        
    # read the CSV file into data frame, using ';' as the separator and the first row as header
    data = pd.read_csv(path_file, sep=";", header=0)
    
    if data['Temperature'].hasnans:
        data_t = data['Temperature'].dropna()
    else:
        data_t = data['Temperature']
        
    if data['Humidity'].hasnans:
        data_h = data['Humidity'].dropna()
    else:
        data_h = data['Humidity']
    
    mp_t = np.asarray(stumpy.stump(data_t, m=60)[:, 0], dtype=float)
    mp_h = np.asarray(stumpy.stump(data_h, m=60)[:, 0], dtype=float)
    
    quantiles_t = np.nanquantile(mp_t, [0.25, 0.75])
    quantiles_h = np.nanquantile(mp_h, [0.25, 0.75])
    
    matrix_profiles.loc[len(matrix_profiles.index)] = [dataset, np.nanmin(mp_t), np.nanmax(mp_t), np.nanmean(mp_t), np.nanstd(mp_t), quantiles_t[0], np.nanmedian(mp_t), quantiles_t[1], np.nanmin(mp_h), np.nanmax(mp_h), np.nanmean(mp_h), np.nanstd(mp_h), quantiles_h[0], np.nanmedian(mp_h), quantiles_h[1]]
    
matrix_profiles = matrix_profiles.transpose()
    
if 'datasets_overview' not in globals():
    datasets_overview = pd.read_csv(FILE_OVERVIEW, sep=";", header=0, index_col=0)
if 'MP_Temperature min' in datasets_overview.index:
    datasets_overview = datasets_overview.drop(index=['MP_Temperature min', 'MP_Temperature max', 'MP_Temperature mean', 'MP_Temperature std', 'MP_Temperature q25', 'MP_Temperature median', 'MP_Temperature q75', 'MP_Humidity min', 'MP_Humidity max', 'MP_Humidity mean', 'MP_Humidity std', 'MP_Humidity q25', 'MP_Humidity median', 'MP_Humidity q75'])
    
datasets_overview = datasets_overview.merge(matrix_profiles, on='Sensor', how='left')
datasets_overview = datasets_overview.reindex(['Sensor', 'Start', 'End', 'Missing Start', 'Missing End', 'Length', 'Temperature min', 'Temperature max', 'Temperature mean', 'Temperature std', 'Temperature q25', 'Temperature median', 'Temperature q75', 'Humidity min', 'Humidity max', 'Humidity mean', 'Humidity std', 'Humidity q25', 'Humidity median', 'Humidity q75', 'Prominence', 'No. Peaks_Temperature', 'No. Peaks_Humidity', 'Peaks_Temperature Height min', 'Peaks_Temperature Height max', 'Peaks_Temperature Height mean', 'Peaks_Temperature Height std', 'Peaks_Temperature Height q25', 'Peaks_Temperature Height median', 'Peaks_Temperature Height q75', 'Peaks_Humidity Height min', 'Peaks_Humidity Height max', 'Peaks_Humidity Height mean', 'Peaks_Humidity Height std', 'Peaks_Humidity Height q25', 'Peaks_Humidity Height median', 'Peaks_Humidity Height q75', 'Peaks_Temperature', 'Peaks_Humidity', 'Peaks_Temperature Start', 'Peaks_Temperature End', 'Peaks_Humidity Start', 'Peaks_Humidity End', 'Peak Prominences Temperature', 'Peak Prominences Humidity', 'MP_Temperature min', 'MP_Temperature max', 'MP_Temperature mean', 'MP_Temperature std', 'MP_Temperature q25', 'MP_Temperature median', 'MP_Temperature q75', 'MP_Humidity min', 'MP_Humidity max', 'MP_Humidity mean', 'MP_Humidity std', 'MP_Humidity q25', 'MP_Humidity median', 'MP_Humidity q75'])
datasets_overview.to_csv(FILE_OVERVIEW, sep=";")